In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE
import os

/home/taerim/miniconda3/envs/FindMyLab/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DIR = "./"
DIR_emb = f"{DIR}/Embeddings/MPnet"
DIR_paper = f"{DIR}/DataCollection"

In [3]:
# Years to analyze
years = list(range(1980, 2026))
print(years)
print(len(years))

# Number of papers to keep per year
# N = 50000 # Actual number uesd for the final project.
N = 1000  # Set a small number for example.

[1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
46


# Read Embeddings

In [4]:
selection_idx = {}
embeddings = []
for year in years:
    year = str(year)
    print(f"Processing year: {year}")
    path = os.path.join(DIR_emb, f"MPnet_embeddings_{year}.npy")
    if os.path.exists(path):
        emb = np.load(path)

        # Randomly select N embeddings if there are more than N
        if len(emb) > N:
            print(f"- Selecting {N} randomly.")
            selection_idx[year] = np.random.choice(len(emb), N, replace=False)
            emb = emb[selection_idx[year]]
        else:
            selection_idx[year] = np.arange(len(emb))
            print(f"- Using all {len(emb)} papers.")
            print("")

        embeddings.append(emb)
    else:
        print(f"File not found: {path}")

Processing year: 1980
- Selecting 1000 randomly.
Processing year: 1981
- Selecting 1000 randomly.
Processing year: 1982
- Selecting 1000 randomly.
Processing year: 1983
- Selecting 1000 randomly.
Processing year: 1984
- Selecting 1000 randomly.
Processing year: 1985
- Selecting 1000 randomly.
Processing year: 1986
- Selecting 1000 randomly.
Processing year: 1987
- Selecting 1000 randomly.
Processing year: 1988
- Selecting 1000 randomly.
Processing year: 1989
- Selecting 1000 randomly.
Processing year: 1990
- Selecting 1000 randomly.
Processing year: 1991
- Selecting 1000 randomly.
Processing year: 1992
- Selecting 1000 randomly.
Processing year: 1993
- Selecting 1000 randomly.
Processing year: 1994
- Selecting 1000 randomly.
Processing year: 1995
- Selecting 1000 randomly.
Processing year: 1996
- Selecting 1000 randomly.
Processing year: 1997
- Selecting 1000 randomly.
Processing year: 1998
- Selecting 1000 randomly.
Processing year: 1999
- Selecting 1000 randomly.
Processing year: 200

In [ ]:
# save dictionary of selection_idx
np.save(f"{DIR}/selection_idx_part1.npy", selection_idx)

# load dic
load_dic = np.load(f"{DIR}/selection_idx_part1.npy", allow_pickle=True).item()

In [6]:
# concat all embeddings
embeddings = np.concatenate(embeddings, axis=0)
print(f"Total embeddings shape: {embeddings.shape}")

Total embeddings shape: (46000, 768)


# Read paper information

In [7]:
df_paper = []
for year in years:
    year = str(year)
    print(f"Processing papers for year: {year}")

    path = os.path.join(DIR_paper, f"pubmed_data_{year}.tsv")
    if os.path.exists(path):
        df_year = pd.read_csv(
            path,
            sep="\t",
            usecols=["PMID", "journal", "Year", "corresponding_countries"],
        )
        # select only the papers that were selected for embeddings
        df_year = df_year.iloc[selection_idx[year]]
        df_paper.append(df_year)
    else:
        print(f"File not found: {path}")

df_paper = pd.concat(df_paper, ignore_index=True)
print(f"Total papers: {len(df_paper)}")

Processing papers for year: 1980
Processing papers for year: 1981
Processing papers for year: 1982
Processing papers for year: 1983
Processing papers for year: 1984
Processing papers for year: 1985
Processing papers for year: 1986
Processing papers for year: 1987
Processing papers for year: 1988
Processing papers for year: 1989
Processing papers for year: 1990
Processing papers for year: 1991
Processing papers for year: 1992
Processing papers for year: 1993
Processing papers for year: 1994
Processing papers for year: 1995
Processing papers for year: 1996
Processing papers for year: 1997
Processing papers for year: 1998
Processing papers for year: 1999
Processing papers for year: 2000
Processing papers for year: 2001
Processing papers for year: 2002
Processing papers for year: 2003
Processing papers for year: 2004
Processing papers for year: 2005
Processing papers for year: 2006
Processing papers for year: 2007
Processing papers for year: 2008
Processing papers for year: 2009
Processing

In [8]:
df_paper.head()

,Year,PMID,journal,corresponding_countries
0,1980,7431027,"Journal of neurology, neurosurgery, and psychi...",NaN
1,1980,6995820,Medical hypotheses,NaN
2,1980,7448697,Cancer,NaN
3,1980,11219864,Carcinogenesis,NaN
4,1980,6779220,Neurology,NaN


# Read ASJC Annotation

In [9]:
df_ASJC = pd.read_csv(f"{DIR}/ASJC_Annotation.txt", sep="\t")
df_ASJC.head()

,Title,ASJC_1st,Subject_area_1st
0,PloS one,Multidisciplinary,Multidisciplinary
1,Scientific reports,Multidisciplinary,Multidisciplinary
2,The Journal of biological chemistry,Molecular Biology,Life Sciences
3,Proceedings of the National Academy of Science...,Multidisciplinary,Multidisciplinary
4,International journal of molecular sciences,Inorganic Chemistry,Physical Sciences


In [10]:
print(df_paper.shape)
df_paper_ASJC = df_paper.merge(df_ASJC, left_on="journal", right_on="Title", how="left")
print(df_paper_ASJC.shape)
df_paper_ASJC.head()

(46000, 4)
(46000, 7)


,Year,PMID,journal,corresponding_countries,Title,ASJC_1st,Subject_area_1st
0,1980,7431027,"Journal of neurology, neurosurgery, and psychi...",NaN,"Journal of neurology, neurosurgery, and psychi...",Surgery,Health Sciences
1,1980,6995820,Medical hypotheses,NaN,Medical hypotheses,Medicine,Health Sciences
2,1980,7448697,Cancer,NaN,Cancer,Cancer Research,Life Sciences
3,1980,11219864,Carcinogenesis,NaN,Carcinogenesis,Cancer Research,Life Sciences
4,1980,6779220,Neurology,NaN,Neurology,Neurology (clinical),Health Sciences


In [11]:
df_paper_ASJC.ASJC_1st.value_counts().head(10)

ASJC_1st
Medicine                                  2980
Biochemistry                              1703
Surgery                                   1388
Molecular Biology                         1233
Multidisciplinary                         1156
Pharmacology                              1021
Genetics                                  1012
Cardiology and Cardiovascular Medicine     961
Radiology                                  902
Public Health                              844
Name: count, dtype: int64

# PCA

In [15]:
# Do PCA
n_PC = 5

pca = PCA(n_components=n_PC)
pca_result = pca.fit_transform(embeddings)
pca_result = pd.DataFrame(pca_result, columns=[f"PC{i+1}" for i in range(n_PC)])
pca_result["PMID"] = df_paper_ASJC.PMID
pca_result = pca_result.merge(
    df_paper_ASJC,
    on="PMID",
    how="left",
)

print(pca_result.shape)
pca_result.head()

(46000, 12)


,PC1,PC2,PC3,PC4,PC5,PMID,Year,journal,corresponding_countries,Title,ASJC_1st,Subject_area_1st
0,-0.052727,0.018738,-0.150800,-0.099476,-0.267547,7431027,1980,"Journal of neurology, neurosurgery, and psychi...",NaN,"Journal of neurology, neurosurgery, and psychi...",Surgery,Health Sciences
1,0.151673,-0.116654,-0.129296,0.123304,-0.179218,6995820,1980,Medical hypotheses,NaN,Medical hypotheses,Medicine,Health Sciences
2,-0.153047,-0.107056,-0.012100,-0.306701,-0.129546,7448697,1980,Cancer,NaN,Cancer,Cancer Research,Life Sciences
3,0.278895,0.049043,0.097265,-0.126300,-0.152348,11219864,1980,Carcinogenesis,NaN,Carcinogenesis,Cancer Research,Life Sciences
4,-0.024570,-0.257223,-0.239953,0.100530,0.046187,6779220,1980,Neurology,NaN,Neurology,Neurology (clinical),Health Sciences


In [ ]:
# save PCA results
pca_result.to_csv(f"{DIR}/Part1_pca_result.txt", index=False, sep="\t")

# t-SNE

In [17]:
# tSNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_result = tsne.fit_transform(embeddings)
tsne_result = pd.DataFrame(tsne_result, columns=["tSNE1", "tSNE2"])
tsne_result["PMID"] = df_paper_ASJC.PMID
tsne_result = tsne_result.merge(
    df_paper_ASJC,
    on="PMID",
    how="left",
)

print(tsne_result.shape)
tsne_result.head()

(46000, 9)


,tSNE1,tSNE2,PMID,Year,journal,corresponding_countries,Title,ASJC_1st,Subject_area_1st
0,-27.704924,23.642826,7431027,1980,"Journal of neurology, neurosurgery, and psychi...",NaN,"Journal of neurology, neurosurgery, and psychi...",Surgery,Health Sciences
1,18.297745,35.370819,6995820,1980,Medical hypotheses,NaN,Medical hypotheses,Medicine,Health Sciences
2,0.353476,-75.841599,7448697,1980,Cancer,NaN,Cancer,Cancer Research,Life Sciences
3,32.061661,-0.633918,11219864,1980,Carcinogenesis,NaN,Carcinogenesis,Cancer Research,Life Sciences
4,-39.023907,53.432934,6779220,1980,Neurology,NaN,Neurology,Neurology (clinical),Health Sciences


In [ ]:
# save tSNE results
tsne_result.to_csv(f"{DIR}/Part1_tsne_result.txt", index=False, sep="\t")